# Loading and downloading known Sentinel-1 scenes using the STAC API

This notebook demonstrates the use of a custom function to query individual Sentinel-1 scene IDs and then either load them as an xarray, or download their bands as GeoTiffs.
This may be valuable if you have existing Sentinel-1 scene-based workflows.

## Set up

### Import required libraries

In [ ]:
from odc.stac import configure_s3_access
from pystac_client import Client
from pathlib import Path
from asf_search import ASFSearchError
from pystac_client.exceptions import APIError

from de_sar_demo.loading import find_and_load_single_scene_from_stac


### Environment set up

In [ ]:
# Set global connection variables
catalog = "https://explorer.dev.dea.ga.gov.au/stac"
stac_client = Client.open(catalog)
configure_s3_access(cloud_defaults=True, aws_unsigned=True)

## Function for querying and loading known scenes from STAC

We have defined a utility function to make it easier to load scenes.
It's contained in [de_sar_demo/loading.py](../../de_sar_demo/loading.py).

The function `find_and_load_single_scene_from_stac` takes a few arguments:
* `stac_client`: the STAC client to use for querying
* `scene`: the scene ID to load, e.g. `S1A_IW_SLC__1SSH_20250402T103324_20250402T103352_058576_073FF3_ED28`. Must be an SLC (there are no GRD scenes available in the collection 0 release).
* `output_format`: one of `"geotiff"` or `"xarray"`. If using `"xarray"`, we recommend loading scenes as needed, rather than many scenes in a loop. If using `"geotiff"`, you can use a loop to download a list of scenes from a .txt file. Default is `"geotiff"`
* `output_dir`: a string indicating where on disk the outputs should be stored if using `"geotiff"`. Default is `None`, which means files are saved to the directory that contains this notebook
* `speckle_filter`: whether to apply the Lee speckle filter with a window of 3 pixels. Default is `True`
* `db`: whether to convert the gamma0 backscatter from linear scale to decibels. Default is `True`

The function will query the Alaska Satellite Facility for the scene, and find key metadata for loading, such as the scene's start and stop times, central latitude and longitude, and polarisation.
The function will also confirm that the requested scene is an SLC (GRD scenes are not available in our current collection 0 data offering).

## Load a list of scenes and save each gamma0 band to a GeoTIFF

The following code block demonstrates how to construct a loop that will try to download each scene, catching API errors (such as connection time outs). 
If a scene fails to download due to a time out, try running the loop again. 
If a scene fails for another reason, it may not be a valid request.

In [ ]:
# Make a place for output files
output_path = "outputs"
Path(output_path).mkdir(exist_ok=True)

scene_list = [
    "S1A_IW_SLC__1SSH_20250417T105815_20250417T105844_058795_0748E8_E17F","S1A_IW_SLC__1SDV_20250629T193037_20250629T193104_059865_076F8D_42C1"
    ]

for scene in scene_list:
    try:
        find_and_load_single_scene_from_stac(
            stac_client,
            scene,
            output_format = "geotiff", 
            output_dir = output_path, 
            speckle_filter=True, 
            db=True
        )
    except ASFSearchError as asf_error:
        raise RuntimeError("    Connection to ASF was unsuccessful. Try again.")
        continue
    except APIError as stac_api_error:
        raise RuntimeError("    Connection to DEA STAC API using pystac-client was unsuccessful. Try again.")
        continue
    except ValueError:
        raise
        continue
    except RuntimeError:
        raise
        continue

## Load a single scenes as an xarray
The function allows loading of a single scene as an xarray. 
We do not recommend using this version in a loop for a large number of scenes, as entire IW SLC scenes at 20m resolution consume a reasonable amount of memory.
However, this version of the function is useful for loading a small number of scenes into individual xarrays, which can then be further manipulated in Python.

In [ ]:
try: 
    ds_83BA = find_and_load_single_scene_from_stac(
        stac_client,
        scene="S1A_IW_SLC__1SSH_20250417T105815_20250417T105844_058795_0748E8_E17F",
        output_format = "xarray", 
        speckle_filter=True, 
        db=True
    )
except ASFSearchError as asf_error:
    raise RuntimeError("    Connection to ASF was unsuccessful. Try again.")
except APIError as stac_api_error:
    raise RuntimeError("    Connection to DEA STAC API using pystac-client was unsuccessful. Try again.")

### View the xarray

In [ ]:
ds_83BA

### Plot the xarray

In [ ]:
ds_83BA["HH_gamma0"].plot.imshow(robust=True, cmap="Greys_r")